# CS2309 — Precision eval (chọn version)

Mỗi lần **chỉ bật 1** config → export `bundle.zip` → so sánh local sau (`jobs_june17.json`).

| Alias | Config |
|---|---|
| `fp32` | `baseline_fp32` |
| `fp16` / `fp8` / `fp4` | compute + EditCache (disk fp32) |
| `fp16_weight` / `fp4_weight` | disk fp16 (+ quant fp4) |

**Workflow 3 lần (cùng runtime / cùng `MAX_JOBS=600`):**

1. **`fp32`** — setup tải Qualcomm (một lần).
2. **`fp16_weight`** — prepare: có fp32 → **tự convert** cây fp16 (nếu chưa có).
3. **`fp4_weight`** — prepare: **reuse** fp16 nếu đã có (không convert lại); eval load fp16 rồi **quant fp4** (bitsandbytes, cần CUDA/T4).

Gộp bundle local: `python scripts/compare_precision_runs.py`.


### ⓪ Chọn config

In [24]:
# T4 16GB: chỉ True MỘT config mỗi lần chạy.
SELECT = {
    "fp32": False,
    "fp16": False,
    "fp8": False,
    "fp4": False,
    "fp16_weight": False,
    "fp4_weight": True,   # lần 3: sau fp16_weight (reuse cây fp16 → quant fp4)
}
EVAL_CONFIGS = ",".join(k for k, on in SELECT.items() if on)
assert EVAL_CONFIGS, "Bật ít nhất 1 config"
assert sum(1 for v in SELECT.values() if v) == 1, "T4: chỉ 1 config / lần"

USE_DRIVE = False
USE_PRIVATE_REPO = True
REPO_SLUG = "NguyenKz/CS2309.CH201"
COLAB_REPO_DIR = "/content/CS2309.CH201"
DRIVE_FP16 = "/content/drive/MyDrive/CS2309/swiftedit_weights_fp16"
DRIVE_FP32 = "/content/drive/MyDrive/CS2309/swiftedit_weights"

_selected = [k for k, on in SELECT.items() if on]
_weight_only = all(k.endswith("_weight") for k in _selected)
# Skip Qualcomm chỉ khi weight-only + Drive có fp16. Còn lại: setup tải/giữ fp32.
SKIP_QUALCOMM_IN_SETUP = bool(USE_DRIVE and _weight_only)
# KHÔNG còn NO_CONVERT — prepare tự quyết: thiếu fp16 + có fp32 → convert.

N_IMAGES = 200
EDITS_PER_IMAGE = 3
MAX_JOBS = 600
print("EVAL_CONFIGS =", EVAL_CONFIGS)
print("MAX_JOBS =", MAX_JOBS)
print("SKIP_QUALCOMM_IN_SETUP =", SKIP_QUALCOMM_IN_SETUP)
if "fp4_weight" in _selected or "fp4" in _selected:
    print("NOTE: fp4 cần CUDA (T4) + bitsandbytes; không chạy trên Mac/MPS", flush=True)


EVAL_CONFIGS = fp16_weight
MAX_JOBS = 600
SKIP_QUALCOMM_IN_SETUP = False


### ① Clone + GPU + token

In [25]:
import getpass, os, subprocess, sys
from pathlib import Path

def _run_stream(cmd, *, cwd=None, env=None, check=True):
    """In log live (Colab không nuốt output)."""
    print("+", " ".join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(
        cmd, cwd=cwd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    print(f"[exit {rc}]", flush=True)
    if check and rc != 0:
        raise RuntimeError(f"Command failed ({rc}): {cmd}")
    return rc

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

COLAB_REPO_DIR = Path(COLAB_REPO_DIR)
DRIVE_FP16 = Path(DRIVE_FP16)
DRIVE_FP32 = Path(DRIVE_FP32)

def _colab_repo_url():
    if not USE_PRIVATE_REPO:
        return f"https://github.com/{REPO_SLUG}.git"
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception as e:
        print("Secrets:", e, flush=True)
    if not token:
        token = getpass.getpass("GITHUB_TOKEN: ").strip()
    if not token:
        raise RuntimeError("Thiếu GITHUB_TOKEN")
    return f"https://{token}@github.com/{REPO_SLUG}.git"

if IN_COLAB:
    r = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
    if r.returncode != 0 or not r.stdout.strip():
        raise RuntimeError("Cần GPU T4")
    print("GPU:", r.stdout.strip(), flush=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    if not (COLAB_REPO_DIR / "SwiftEdit" / "infer.py").exists():
        _run_stream(["git", "clone", "--depth", "1", _colab_repo_url(), str(COLAB_REPO_DIR)])
    else:
        _run_stream(["git", "-C", str(COLAB_REPO_DIR), "pull", "--ff-only"], check=False)
    PROJECT_ROOT = COLAB_REPO_DIR
    os.chdir(PROJECT_ROOT)
    os.environ.setdefault("HF_HOME", "/content/huggingface")
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent
print("PROJECT_ROOT", PROJECT_ROOT, flush=True)


GPU: Tesla T4
+ git -C /content/CS2309.CH201 pull --ff-only
From https://github.com/NguyenKz/CS2309.CH201
   52303f8..e87acaf  main       -> origin/main
Updating 52303f8..e87acaf
Fast-forward
 .../CS2309_SwiftEdit_precision_disk_vram.ipynb     | 313 +++++++--------------
 1 file changed, 109 insertions(+), 204 deletions(-)
[exit 0]
PROJECT_ROOT /content/CS2309.CH201


### ② Setup (pip + HF + Qualcomm nếu cần)

In [26]:
env = os.environ.copy()
env["COLAB_REPO_DIR"] = str(PROJECT_ROOT) if IN_COLAB else ""
env["PYTHONUNBUFFERED"] = "1"
if IN_COLAB and SKIP_QUALCOMM_IN_SETUP:
    env["SWIFTEDIT_SKIP_WEIGHTS_DOWNLOAD"] = "1"
    print("SKIP Qualcomm trong setup (Drive *_weight)", flush=True)
else:
    print("Setup sẽ tải/verify Qualcomm nếu chưa có", flush=True)

setup_sh = PROJECT_ROOT / "scripts" / ("setup_colab.sh" if IN_COLAB else "setup_macos.sh")
_run_stream(["bash", str(setup_sh)], cwd=PROJECT_ROOT, env=env)
_run_stream([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "torchmetrics", "pyarrow"])
if "fp4_weight" in _selected or "fp4" in _selected:
    print("bitsandbytes OK (cần cho quant fp4)", flush=True)
    try:
        import bitsandbytes as bnb
        print("bitsandbytes", getattr(bnb, "__version__", "?"), flush=True)
    except Exception as e:
        raise RuntimeError(f"Thiếu bitsandbytes cho fp4: {e}")

se = PROJECT_ROOT / "SwiftEdit"
removed = []
for p in list(se.glob("swiftedit_weights.tar.gz*")):
    print("rm leftover", p.name, p.stat().st_size // (1024**2), "MB", flush=True)
    p.unlink(missing_ok=True)
    removed.append(p.name)
print("Removed archives:", removed or "(none)", flush=True)
WP32 = se / "swiftedit_weights"
WP16 = se / "swiftedit_weights_fp16"
print("fp32 ready?", (WP32 / "sbv2_0.5").is_dir(), flush=True)
print("fp16 ready?", (WP16 / "sbv2_0.5").is_dir(), flush=True)


Setup sẽ tải/verify Qualcomm nếu chưa có
+ bash /content/CS2309.CH201/scripts/setup_colab.sh
[setup] Colab setup (bước 2 — cần clone trước)
[setup] GPU: Tesla T4
[setup] pip install...
  transformers: 4.57.6 (>= 4.46.0)
  diffusers: 0.35.2 (>= 0.32.0)
  numpy: 2.0.2
  torchvision: 0.26.0+cu128
  diffusers + transformers OK
[setup] swiftedit_weights OK
[setup] Tải HF models...
1/3 SD 2.1 aux (scheduler, vae, tokenizer, text_encoder)...

Fetching 15 files: 100%|██████████| 15/15 [00:00<00:00, 98457.84it/s]
2/3 sd-turbo inversion (scheduler, vae, tokenizer, text_encoder)...

Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 52132.59it/s]
3/3 IP-Adapter image_encoder...
done.
[setup] Wrote /content/CS2309.CH201/.setup_env
  torch: 2.11.0+cu128
  cuda: True
  GPU: Tesla T4
  mps: False
[setup] done — PROJECT_ROOT=/content/CS2309.CH201
[exit 0]
+ /usr/bin/python3 -m pip install -q bitsandbytes torchmetrics pyarrow
[exit 0]
Removed archives: (none)
fp32 ready? True
fp16 ready? False


### ②b Dataset June17

In [27]:
auto = PROJECT_ROOT / "data" / "PIE-Bench-auto200"
if not (auto / "annotation_images").is_dir():
    print("Freeze auto200...", flush=True)
    _run_stream([sys.executable, "-u", str(PROJECT_ROOT / "scripts" / "freeze_piebench_auto200.py")], cwd=PROJECT_ROOT)
_run_stream([sys.executable, "-u", str(PROJECT_ROOT / "scripts" / "build_june17_jobs.py")], cwd=PROJECT_ROOT)
print("jobs_june17:", (PROJECT_ROOT / "data" / "jobs_june17.json").is_file(), flush=True)


+ /usr/bin/python3 -u /content/CS2309.CH201/scripts/build_june17_jobs.py
Wrote /content/CS2309.CH201/data/PIE-Bench-auto200/jobs_june17.json (600 jobs = 200 × 3)
Also wrote commit-friendly /content/CS2309.CH201/data/jobs_june17.json
[exit 0]
jobs_june17: True


### ③ Prepare weights (tự động)

- Có Drive fp16 → symlink
- Có fp32 local, thiếu fp16, config `*_weight` → **convert** (không cần flag)
- **`fp4_weight`:** nếu fp16 đã có (sau lần 2) → chỉ verify, không convert lại
- Không tải Qualcomm ở đây nếu setup đã có fp32


In [ ]:
WP32 = PROJECT_ROOT / "SwiftEdit" / "swiftedit_weights"
WP16 = PROJECT_ROOT / "SwiftEdit" / "swiftedit_weights_fp16"
cmd = [
    sys.executable, "-u",
    str(PROJECT_ROOT / "scripts" / "prepare_colab_weights.py"),
    "--configs", EVAL_CONFIGS,
    "--drive-fp16", str(DRIVE_FP16),
    "--drive-fp32", str(DRIVE_FP32),
    "--local-fp32", str(WP32),
    "--local-fp16", str(WP16),
    # Không truyền --no-convert: *_weight + đã có fp32 → tự convert
]
print("fp32?", (WP32 / "sbv2_0.5").is_dir(), "fp16?", (WP16 / "sbv2_0.5").is_dir(), flush=True)
_run_stream(cmd, cwd=PROJECT_ROOT)
print("weights OK", flush=True)
print("fp16:", (WP16 / "sbv2_0.5").is_dir(), "fp32:", (WP32 / "sbv2_0.5").is_dir(), flush=True)


fp32? True fp16? False
+ /usr/bin/python3 -u /content/CS2309.CH201/scripts/prepare_colab_weights.py --configs fp16_weight --drive-fp16 /content/drive/MyDrive/CS2309/swiftedit_weights_fp16 --drive-fp32 /content/drive/MyDrive/CS2309/swiftedit_weights --local-fp32 /content/CS2309.CH201/SwiftEdit/swiftedit_weights --local-fp16 /content/CS2309.CH201/SwiftEdit/swiftedit_weights_fp16
configs: fp16_disk
need_fp32=False need_fp16_disk=True
[fp16] Drive chưa có: /content/drive/MyDrive/CS2309/swiftedit_weights_fp16
fp16 chưa sẵn sàng trên local/Drive.
fp32 local OK: /content/CS2309.CH201/SwiftEdit/swiftedit_weights
Convert fp32 → fp16 trên máy này (lâu / tốn RAM). Khuyên convert Mac + Drive.
2026-07-19 11:43:08.740280: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compil

### ④ Eval → bundle.zip

In [ ]:
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["OMP_NUM_THREADS"] = "1"
env["MKL_NUM_THREADS"] = "1"
env["OPENBLAS_NUM_THREADS"] = "1"
cmd = [
    sys.executable, "-u",
    str(PROJECT_ROOT / "scripts" / "run_precision_disk_vram_eval.py"),
    "--configs", EVAL_CONFIGS,
    "--n-images", str(N_IMAGES),
    "--edits-per-image", str(EDITS_PER_IMAGE),
    "--jobs-manifest", str(PROJECT_ROOT / "data" / "jobs_june17.json"),
    "--weights-fp32", str(WP32),
    "--weights-fp16", str(WP16),
]
if MAX_JOBS is not None:
    cmd += ["--max-jobs", str(MAX_JOBS)]
_run_stream(cmd, cwd=PROJECT_ROOT, env=env)
bundles = sorted((PROJECT_ROOT / "experimental_data").glob(f"precision_run_*/bundle_{EVAL_CONFIGS}.zip"))
print("Latest bundles:", bundles[-3:], flush=True)
if IN_COLAB and bundles:
    from google.colab import files
    files.download(str(bundles[-1]))


### Sau khi đủ bundle (local)

```bash
python scripts/compare_precision_runs.py
```
